# Extra Baselines

Compute graph-native baseline risk scores, tune a train-set threshold, and evaluate the fixed threshold on the test graphs.

In [41]:
from pathlib import Path
import pickle
import sys

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT.name == "code":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "code"))
sys.path.append(str(PROJECT_ROOT / "code" / "networks"))

from extra_baselines import BASELINE_NAMES, baseline_graphs_to_frame
from utils import get_product_list

BASELINE_NAMES

('inverse_partner_outdegree', 'import_concentration_hhi')

In [ ]:
digits = 2
graph_type = "export"  # "export" or "total"
multi_graph = True  # True or False

# Edge values are log-transformed when the graph CSVs are generated.
weight_transform = "exp"

if multi_graph:
    graphs_root = PROJECT_ROOT / "data" / "5. Graphs Data" / f"{digits}_digits" / "multi-graph" / graph_type
else:
    graphs_root = PROJECT_ROOT / "data" / "5. Graphs Data" / f"{digits}_digits" / graph_type

graphs_root

WindowsPath('c:/Users/ginof/OneDrive/Documents/GitHub/Detecting-Global-Trade-Vulnerabilities-via-Graph-Neural-Networks/data/5. Graphs Data/2_digits/export')

In [43]:
def graph_metadata(digits, graph_type, multi_graph, split):
    if split == "train":
        years = list(range(2012, 2021))
    elif split == "test":
        years = [2021]
    else:
        raise ValueError("split must be 'train' or 'test'.")

    if multi_graph:
        return [
            {
                "digits": digits,
                "year": year,
                "graph_type": graph_type,
                "commodity": "all",
            }
            for year in years
        ]

    products = get_product_list(digits=digits)
    return [
        {
            "digits": digits,
            "year": year,
            "graph_type": graph_type,
            "commodity": commodity,
        }
        for commodity in products
        for year in years
    ]


def node_features_path(graphs_root, meta, multi_graph):
    if multi_graph:
        filename = f"node_features-{meta['year']}-{meta['graph_type']}.csv"
    else:
        filename = f"node_features-{meta['year']}-{meta['commodity']}-{meta['graph_type']}.csv"
    return graphs_root / filename


def country_ids_for_graphs(graphs_root, metadata, multi_graph):
    country_ids = []
    for meta in metadata:
        node_path = node_features_path(graphs_root, meta, multi_graph)
        country_ids.append(pd.read_csv(node_path, usecols=["country_id"])["country_id"].tolist())
    return country_ids


def load_graph_split(graphs_root, split):
    graphs_path = graphs_root / f"{split}-graphs.pkl"
    with graphs_path.open("rb") as f:
        graphs = pickle.load(f)
    return graphs


def baseline_split_frame(split, baseline):
    graphs = load_graph_split(graphs_root, split)
    metadata = graph_metadata(digits, graph_type, multi_graph, split)
    country_ids_by_graph = country_ids_for_graphs(graphs_root, metadata, multi_graph)

    assert len(metadata) == len(graphs), (len(metadata), len(graphs))

    df = baseline_graphs_to_frame(
        graphs,
        country_ids_by_graph,
        metadata,
        baseline=baseline,
        weight_transform=weight_transform,
    )
    return df


train_dfs = {baseline: baseline_split_frame("train", baseline) for baseline in BASELINE_NAMES}
test_dfs = {baseline: baseline_split_frame("test", baseline) for baseline in BASELINE_NAMES}

{baseline: (train_dfs[baseline].shape, test_dfs[baseline].shape) for baseline in BASELINE_NAMES}

Loading DATA_PATHS.yaml from C:\Users\ginof\OneDrive\Documents\GitHub\Detecting-Global-Trade-Vulnerabilities-via-Graph-Neural-Networks\data\DATA_PATHS.yaml
Loading DATA_PATHS.yaml from C:\Users\ginof\OneDrive\Documents\GitHub\Detecting-Global-Trade-Vulnerabilities-via-Graph-Neural-Networks\data\DATA_PATHS.yaml
Loading DATA_PATHS.yaml from C:\Users\ginof\OneDrive\Documents\GitHub\Detecting-Global-Trade-Vulnerabilities-via-Graph-Neural-Networks\data\DATA_PATHS.yaml
Loading DATA_PATHS.yaml from C:\Users\ginof\OneDrive\Documents\GitHub\Detecting-Global-Trade-Vulnerabilities-via-Graph-Neural-Networks\data\DATA_PATHS.yaml


{'inverse_partner_outdegree': ((192943, 7), (21441, 7)),
 'import_concentration_hhi': ((192943, 7), (21441, 7))}

In [44]:
train_dfs

{'inverse_partner_outdegree':         digits  year graph_type commodity  country_id  y          risk
 0            2  2012     export        01         533  0  5.268777e-09
 1            2  2012     export        01           4  0  6.607721e-08
 2            2  2012     export        01          24  0  4.686492e-09
 3            2  2012     export        01           8  0  5.347008e-08
 4            2  2012     export        01          20  0  2.417066e-09
 ...        ...   ...        ...       ...         ... ..           ...
 192938       2  2020     export        97         710  1  6.615407e-08
 192939       2  2020     export        97         894  0  1.630687e-07
 192940       2  2020     export        97         716  0  6.999819e-07
 192941       2  2020     export        97         158  0  3.873564e-09
 192942       2  2020     export        97         999  0  2.701697e-07
 
 [192943 rows x 7 columns],
 'import_concentration_hhi':         digits  year graph_type commodity  count

In [45]:
def threshold_candidates(risk, n_quantiles=500):
    risk = np.asarray(risk, dtype=float)
    risk = risk[np.isfinite(risk)]
    if risk.size == 0:
        raise ValueError("No finite risk values available for threshold tuning.")

    candidates = np.unique(np.quantile(risk, np.linspace(0, 1, n_quantiles)))
    candidates = np.r_[-np.inf, candidates, np.inf]
    return candidates


def tune_threshold(train_df):
    y_true = train_df["y"].to_numpy()
    risk = train_df["risk"].to_numpy()

    rows = []
    for threshold in threshold_candidates(risk):
        y_pred = (risk >= threshold).astype(int)
        rows.append(
            {
                "threshold": threshold,
                "f1_positive": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
                "precision_positive": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
                "recall_positive": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
            }
        )

    scores = pd.DataFrame(rows)
    best = scores.sort_values(
        ["f1_positive", "recall_positive", "precision_positive"],
        ascending=False,
    ).iloc[0]
    return float(best["threshold"]), scores


def evaluate_threshold(df, threshold):
    y_true = df["y"].to_numpy()
    y_pred = (df["risk"].to_numpy() >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "threshold": threshold,
        "f1_positive": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "precision_positive": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall_positive": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "n": len(df),
        "positive_rate": y_true.mean(),
        "predicted_positive_rate": y_pred.mean(),
    }

In [46]:
thresholds = {}
train_threshold_scores = {}
metric_rows = []

for baseline in BASELINE_NAMES:
    threshold, scores = tune_threshold(train_dfs[baseline])
    thresholds[baseline] = threshold
    train_threshold_scores[baseline] = scores

    train_metrics = evaluate_threshold(train_dfs[baseline], threshold)
    test_metrics = evaluate_threshold(test_dfs[baseline], threshold)

    metric_rows.append({"baseline": baseline, "split": "train", **train_metrics})
    metric_rows.append({"baseline": baseline, "split": "test", **test_metrics})

metrics_df = pd.DataFrame(metric_rows)
metrics_df

,baseline,split,threshold,f1_positive,precision_positive,recall_positive,accuracy,tn,fp,fn,tp,n,positive_rate,predicted_positive_rate
0,inverse_partner_outdegree,train,9.308179e-09,0.053500,0.028367,0.469264,0.617825,117121,71381,2357,2084,192943,0.023017,0.380760
1,inverse_partner_outdegree,test,9.308179e-09,0.073291,0.039980,0.439394,0.623758,13055,7660,407,319,21441,0.033860,0.372137
2,import_concentration_hhi,train,5.547500e-02,0.045372,0.023214,0.998424,0.032968,1927,186575,7,4434,192943,0.023017,0.989976
3,import_concentration_hhi,test,5.547500e-02,0.066220,0.034244,1.000000,0.045054,240,20475,0,726,21441,0.033860,0.988806


In [47]:
predicted_test_dfs = {}

for baseline in BASELINE_NAMES:
    df = test_dfs[baseline].copy()
    df["threshold"] = thresholds[baseline]
    df["y_pred"] = (df["risk"] >= thresholds[baseline]).astype(int)
    predicted_test_dfs[baseline] = df

predicted_test_dfs["inverse_partner_outdegree"].head()

,digits,year,graph_type,commodity,country_id,y,risk,threshold,y_pred
0,2,2021,export,01,533,0,2.215576e-06,9.308179e-09,1
1,2,2021,export,01,4,0,1.064080e-07,9.308179e-09,1
2,2,2021,export,01,24,0,1.785402e-08,9.308179e-09,1
3,2,2021,export,01,660,0,8.271161e-07,9.308179e-09,1
4,2,2021,export,01,8,0,4.998977e-08,9.308179e-09,1


In [48]:
# Raw risk DataFrames keep the requested schema.
expected_columns = ["digits", "year", "graph_type", "commodity", "country_id", "y", "risk"]

for baseline in BASELINE_NAMES:
    assert train_dfs[baseline].columns.tolist() == expected_columns
    assert test_dfs[baseline].columns.tolist() == expected_columns

train_dfs["inverse_partner_outdegree"].head()

,digits,year,graph_type,commodity,country_id,y,risk
0,2,2012,export,01,533,0,5.268777e-09
1,2,2012,export,01,4,0,6.607721e-08
2,2,2012,export,01,24,0,4.686492e-09
3,2,2012,export,01,8,0,5.347008e-08
4,2,2012,export,01,20,0,2.417066e-09


In [49]:
metrics_df.to_csv(f"extra_baseline_metrics-{'mg-' if multi_graph else ''}{graph_type}_{digits}.csv", index=False)